<a href="https://colab.research.google.com/github/Shaifali07/LLM-projects/blob/main/QuestionBankMaker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installations:

In [1]:
!pip install pypdf
!pip install langchain_community
!pip install langchain
!pip install langchain-text-splitters
!pip install langchain_huggingface
!pip install langchain_chroma
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━

Imports

In [2]:
from pypdf import PdfReader
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import re

Extracting Questions from previous papers

In [3]:
DOC_PATH = "/content/cn 1st int 2024.pdf"
loader = PyPDFLoader(DOC_PATH)
pages = loader.load()

In [4]:
text = "\n".join([p.page_content for p in pages])


# Step 1: Normalize text
text = re.sub(r'\n+', ' ', text)

# Step 2: Split by question labels (a), (b), (c)
parts = re.split(r'\([a-h]\)', text)

questions = []

for part in parts:
    part = part.strip()
    if not part:
        continue

    # Extract marks
    marks_match = re.search(r'\[(\d+)\]', part)
    marks = int(marks_match.group(1)) if marks_match else None

    # Remove noise
    clean = re.sub(r'\[\d+\]', '', part)  # remove [2]
    clean = re.sub(r'CO\d+\s*[A,N,E,R,U,C]', '', clean)  # remove CO2 A
    clean = clean.strip()


    questions.append({
        "question": clean,
        "marks": marks,

    })

# Print result
for q in questions:
    print(q)

{'question': 'DHARMSINH DESAI UNIVERSITY, NADIAD  FACULTY OF TECHNOLOGY  FIRST SESSIONAL  SUBJECT: COMPUTER NETWORKS  Page 1 of 2   Examination : B.Tech Semester VI Seat No. :  Date : 02/01/2024 Day :Tuesday  Time :  1hr 15 mins Max. Marks : 36      Q.1       Do as directed.', 'marks': 12}
{'question': 'What is sender’s window in case of selective repeat, go back n, and stop and wait  protocol if the sequence number is 4 bits long?', 'marks': 2}
{'question': 'How many bit(s) error can be detected and corrected by the octets: 0xFF, 0x47 , 0x00  and 0x95?', 'marks': 2}
{'question': 'The receiver receives code word 110010001. The generator polynomial is x 3 + x + 1.  Does the received data have an error? (show the procedure)', 'marks': 2}
{'question': 'A supernet is created by combining some class C blocks. It has a first address of  205.16.32.0 and a supernet mask of 255.255.248.0. How many Class C blocks are there  in this supernet? Explain.', 'marks': 2}
{'question': 'An IPv4 packet ha

In [5]:
from langchain_core.documents import Document

docs = []
data= questions

for i, q in enumerate(data):
    content = f"""
    Question: {q['question']}


    """

    doc = Document(
        page_content=content,
        metadata={
            "marks": q.get("marks")

        }
    )
    print(doc)
    docs.append(doc)

page_content='
    Question: DHARMSINH DESAI UNIVERSITY, NADIAD  FACULTY OF TECHNOLOGY  FIRST SESSIONAL  SUBJECT: COMPUTER NETWORKS  Page 1 of 2   Examination : B.Tech Semester VI Seat No. :  Date : 02/01/2024 Day :Tuesday  Time :  1hr 15 mins Max. Marks : 36      Q.1       Do as directed.


    ' metadata={'marks': 12}
page_content='
    Question: What is sender’s window in case of selective repeat, go back n, and stop and wait  protocol if the sequence number is 4 bits long?


    ' metadata={'marks': 2}
page_content='
    Question: How many bit(s) error can be detected and corrected by the octets: 0xFF, 0x47 , 0x00  and 0x95?


    ' metadata={'marks': 2}
page_content='
    Question: The receiver receives code word 110010001. The generator polynomial is x 3 + x + 1.  Does the received data have an error? (show the procedure)


    ' metadata={'marks': 2}
page_content='
    Question: A supernet is created by combining some class C blocks. It has a first address of  205.16.32.0 and a 

Embeddings

In [6]:
embedings=HuggingFaceEmbeddings()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
persist_directory="/content/db"
vector_store=Chroma.from_documents(
    documents=docs,
    embedding=embedings,
    persist_directory=persist_directory
)

In [27]:
total_docs=5
retriever=vector_store.as_retriever(
      search_kwargs={"k":total_docs}
)


LLM setup

In [9]:
from google.colab import userdata
import os

groq_api_key = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = groq_api_key

In [10]:
llm=ChatGroq(
    model_name="openai/gpt-oss-120b",
    temperature=0)

In [20]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert academic assistant. Your task is to extract ONLY those exam questions from the given text that are relevant to the specified topic."),
    ("human",'''Generate a university-level question bank form the {context} on the following topic: {topic}

Instructions:
1. Identify and extract only valid questions.
2. Ignore:
   - headers (university name, exam info)
   - instructions (e.g., "Attempt any two")
   - page numbers or formatting text
3. Clean and rewrite questions properly (fix grammar if needed).
4. If a question has sub-parts (i), (ii), keep them together.
5. If a question is NOT related to the given topic, ignore it.
6. Do NOT invent new questions.

Output format (strict JSON list):
[

    "question": "...",
    "topic": "{topic}"

]
'''
    )])
chain = prompt | llm

In [24]:
topic='error detection'

In [28]:
docs = retriever.invoke(topic)
context = "\n".join([d.page_content for d in docs])

In [29]:
result = chain.invoke({"topic": topic,"context":context})
print(result)

content='[\n  {\n    "question": "What is error control, where is it provided, where is it mandatory, and what provisions are necessary for error detection?",\n    "topic": "error detection"\n  },\n  {\n    "question": "How many bit errors can be detected and corrected by each of the following octets: 0xFF, 0x47, 0x00, and 0x95?",\n    "topic": "error detection"\n  },\n  {\n    "question": "A receiver obtains the code word 110010001. Using the generator polynomial x^3 + x + 1, determine whether the received data contains an error, showing the procedure.",\n    "topic": "error detection"\n  }\n]' additional_kwargs={'reasoning_content': 'We need to extract only questions relevant to "error detection". The given list includes several questions. Let\'s examine each:\n\n1. "What is error control? Where is it provided and where is it mandatory? What provisions are necessary for error detection?" This is about error detection (part of error control). So include.\n\n2. "How many bit(s) error c

In [ ]:
# def read_pdf_content(pdf_file_path):
#   full_text = ""
#   with open(pdf_file_path, 'rb') as file:
#         reader = PdfReader(file)
#         for page in reader.pages:
#             text = page.extract_text()
#             if text:
#                 full_text += text + "\n"
#   return full_text


In [ ]:
# file_path = '/content/cn 1st int 2024.pdf'
# extracted_text = read_pdf_content(file_path)
# print(extracted_text)

In [ ]:
# query="ip"
# result=vector_store.similarity_search(query,k=1)
# for i in range(len(result)):
#   print("source:" ,result[i].metadata.get('source','unknown'))
#   print(result[i].page_content)


In [ ]:
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
# chunks = text_splitter.split_documents(pages)
# print(chunks[1].page_content)